# LeetCode 1154 — Day of the Year bằng Python

**Link đề:** https://leetcode.com/problems/day-of-the-year/
**Độ khó:** Easy · **Chủ đề:** Math, String

```python
class Solution:
    def dayOfYear(self, date: str) -> int:
        ...
```

## Mục tiêu học
1. Tách chuỗi ngày `"YYYY-MM-DD"` thành ba số nguyên bằng **slicing** và `split("-")`.
2. Nắm chắc **quy tắc năm nhuận** của lịch Gregorian (chia hết cho 4, trừ năm thế kỷ không chia hết cho 400 — ví dụ **1900 không nhuận**).
3. Tính "ngày thứ mấy trong năm" = **tổng số ngày của các tháng trước** + ngày hiện tại.
4. Biết thêm cách dùng **prefix sum** (tổng tiền tố) và thư viện chuẩn `datetime` để đối chiếu.
5. Trace, chứng minh, test, lỗi hay gặp, biến thể và bài tập.

## 1. Đề bài và ví dụ

Cho chuỗi `date` theo định dạng `YYYY-MM-DD` (lịch Gregorian). Trả về **số thứ tự của ngày đó trong năm** (ngày 1/1 là ngày 1).

| `date` | Output | Giải thích |
|---|---:|---|
| `"2019-01-09"` | `9` | ngày 9 tháng 1 |
| `"2019-02-10"` | `41` | 31 ngày tháng 1 + 10 |
| `"2004-03-01"` | `61` | 31 + 29 (2004 nhuận) + 1 |
| `"1900-03-01"` | `60` | 31 + 28 (1900 **không** nhuận) + 1 |

**Constraints:**
- `date.length == 10`
- `date[4] == date[7] == '-'`, các ký tự còn lại là chữ số.
- `date` là ngày hợp lệ từ 1/1/1900 tới 31/12/2019.

In [1]:
date = "2019-02-10"
print(len(date), date[4], date[7])
print("năm:", date[:4], "| tháng:", date[5:7], "| ngày:", date[8:])

10 - -
năm: 2019 | tháng: 02 | ngày: 10


## 2. Nền tảng Python cần biết

### 2.1. Slicing chuỗi `s[a:b]`

Lấy các ký tự từ index `a` tới `b - 1`. Với `"2019-02-10"`:

| index | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| ký tự | 2 | 0 | 1 | 9 | - | 0 | 2 | - | 1 | 0 |

- `date[:4]` → `"2019"` (index 0..3)
- `date[5:7]` → `"02"`
- `date[8:]` → `"10"` (tới hết)

In [2]:
date = "2019-02-10"
for i, ch in enumerate(date):
    print(i, ch, end=" | ")
print()
print(repr(date[:4]), repr(date[5:7]), repr(date[8:]))

0 2 | 1 0 | 2 1 | 3 9 | 4 - | 5 0 | 6 2 | 7 - | 8 1 | 9 0 | 
'2019' '02' '10'


### 2.2. `int("02")` — đổi chuỗi chữ số thành số

`int` bỏ qua số 0 ở đầu: `int("02") = 2`, `int("0009") = 9`. Đừng so sánh chuỗi `"02" < "10"` thay cho số — có lúc đúng có lúc sai.

In [3]:
print(int("02"), int("10"), int("0009"))
print("9" < "10")      # so sánh chuỗi theo từng ký tự: '9' > '1' → False
print(9 < 10)          # so sánh số → True

2 10 9
False
True


### 2.3. `split("-")` và `map(int, ...)`

- `"2019-02-10".split("-")` → `["2019", "02", "10"]`.
- `map(int, list_chuoi)` áp `int` cho từng phần tử.
- Gán đồng thời: `y, m, d = map(int, date.split("-"))`.

In [4]:
parts = "2019-02-10".split("-")
print(parts)
y, m, d = map(int, parts)
print(y, m, d, type(y))

['2019', '02', '10']
2019 2 10 <class 'int'>


### 2.4. Năm nhuận (leap year) — quy tắc Gregorian

Một năm nhuận (tháng 2 có 29 ngày) nếu:
- chia hết cho **4**, **và**
- **không** chia hết cho **100**, **trừ khi** chia hết cho **400**.

Viết bằng Python:
```python
leap = (y % 4 == 0 and y % 100 != 0) or y % 400 == 0
```

| năm | %4 | %100 | %400 | nhuận? |
|---|---|---|---|---|
| 2019 | 3 | | | Không |
| 2016 | 0 | 16 | | Có |
| 1900 | 0 | 0 | 300 | **Không** |
| 2000 | 0 | 0 | 0 | Có |

In [5]:
def is_leap(y: int) -> bool:
    return (y % 4 == 0 and y % 100 != 0) or y % 400 == 0

for y in [2019, 2016, 1900, 2000, 2004, 2100]:
    print(y, is_leap(y))

2019 False
2016 True
1900 False
2000 True
2004 True
2100 False


Bài này có năm 1900 trong miền input — **chính là bẫy** cho ai chỉ kiểm tra `y % 4 == 0`.

### 2.5. List số ngày mỗi tháng

```python
days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
```
Index 0 là tháng 1, index 1 là tháng 2… → tháng `m` ở index `m - 1`.

`sum(lst[:k])` = tổng `k` phần tử đầu.

In [6]:
days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
print(len(days_in_month), sum(days_in_month))   # 12 365
m = 3
print("các tháng trước tháng 3:", days_in_month[:m - 1], "tổng =", sum(days_in_month[:m - 1]))

12 365
các tháng trước tháng 3: [31, 28] tổng = 59


## 3. Cách 1 — Trực giác: đếm từng ngày từ 1/1

Người mới hay nghĩ: "đi từ 1/1, mỗi lần +1 ngày, đến khi gặp ngày cần tìm". Ta mô phỏng bằng 2 vòng lặp (tháng, ngày). Đúng nhưng làm việc thừa: tới 365 bước.

In [7]:
def day_of_year_walk(date: str) -> int:
    y, m, d = map(int, date.split("-"))
    days = [31, 29 if is_leap(y) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    count = 0
    for month in range(1, 13):
        for day in range(1, days[month - 1] + 1):
            count += 1
            if month == m and day == d:
                return count
    return -1   # không xảy ra với input hợp lệ

print(day_of_year_walk("2019-01-09"), day_of_year_walk("2019-02-10"))

9 41


**Nút thắt:** ta không cần đếm từng ngày của các tháng đã qua — chỉ cần **cộng nguyên tháng**: tháng 1 đóng góp 31, tháng 2 đóng góp 28/29…

## 4. Insight

```text
day_of_year = (số ngày của tháng 1 .. tháng m-1) + d
```

Với `"2019-02-10"`: tháng trước là tháng 1 = 31 ngày → `31 + 10 = 41`.
Chỉ có một điểm đặc biệt: tháng 2 = 29 nếu năm nhuận.

In [8]:
y, m, d = 2004, 3, 1
days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
if is_leap(y):
    days[1] = 29
print("tháng trước:", days[:m - 1], "→", sum(days[:m - 1]), "+", d, "=", sum(days[:m - 1]) + d)

tháng trước: [31, 29] → 60 + 1 = 61


## 5. Cách 2 — Lời giải chính: cộng các tháng trước

**Mổ xẻ:**
1. Tách `y, m, d`.
2. Tạo list số ngày mỗi tháng; nếu nhuận thì sửa `days[1] = 29`.
3. Trả `sum(days[:m - 1]) + d`.

Lưu ý: tạo list **mới mỗi lần gọi** (không sửa list dùng chung), nếu không lần gọi năm nhuận sẽ làm hỏng lần gọi năm thường sau đó.

In [9]:
def day_of_year(date: str) -> int:
    y, m, d = int(date[:4]), int(date[5:7]), int(date[8:])
    days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    if is_leap(y):
        days[1] = 29
    return sum(days[:m - 1]) + d

for s in ["2019-01-09", "2019-02-10", "2004-03-01", "1900-03-01", "2019-12-31", "2000-12-31"]:
    print(s, "->", day_of_year(s))

2019-01-09 -> 9
2019-02-10 -> 41
2004-03-01 -> 61
1900-03-01 -> 60
2019-12-31 -> 365
2000-12-31 -> 366


## 6. TRACE từng bước

Trace với `"2016-05-20"` (2016 nhuận):

| tháng | số ngày | tổng tích lũy trước khi cộng | sau khi cộng |
|---:|---:|---:|---:|
| 1 | 31 | 0 | 31 |
| 2 | 29 | 31 | 60 |
| 3 | 31 | 60 | 91 |
| 4 | 30 | 91 | 121 |
| + ngày | 20 | 121 | **141** |

In [10]:
def trace_day_of_year(date: str) -> int:
    y, m, d = map(int, date.split("-"))
    leap = is_leap(y)
    print(f"date={date}: y={y}, m={m}, d={d}, nhuận={leap}")
    days = [31, 29 if leap else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    total = 0
    for month in range(1, m):
        before = total
        total += days[month - 1]
        print(f"  cộng tháng {month:>2} ({days[month - 1]} ngày): {before} -> {total}")
    print(f"  cộng ngày {d}: {total} -> {total + d}")
    return total + d

trace_day_of_year("2016-05-20")
print()
trace_day_of_year("1900-03-01")
print()
trace_day_of_year("2019-01-09")

date=2016-05-20: y=2016, m=5, d=20, nhuận=True
  cộng tháng  1 (31 ngày): 0 -> 31
  cộng tháng  2 (29 ngày): 31 -> 60
  cộng tháng  3 (31 ngày): 60 -> 91
  cộng tháng  4 (30 ngày): 91 -> 121
  cộng ngày 20: 121 -> 141

date=1900-03-01: y=1900, m=3, d=1, nhuận=False
  cộng tháng  1 (31 ngày): 0 -> 31
  cộng tháng  2 (28 ngày): 31 -> 59
  cộng ngày 1: 59 -> 60

date=2019-01-09: y=2019, m=1, d=9, nhuận=False
  cộng ngày 9: 0 -> 9


9

Với tháng 1, vòng `range(1, 1)` **rỗng** → không cộng tháng nào, kết quả chỉ là `d`. Đây là biên tự nhiên, không cần `if` riêng.

## 7. Cách 3 — Prefix sum (bảng tích lũy tính sẵn)

Thay vì `sum(days[:m-1])` mỗi lần (tối đa 11 phép cộng), ta tính trước bảng `before[m]` = số ngày trước tháng `m` (năm thường):

```text
before = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
```

Kết quả = `before[m - 1] + d + (1 nếu nhuận và m > 2)`.

In [11]:
from itertools import accumulate

days_normal = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
before = [0] + list(accumulate(days_normal))[:-1]
print(list(accumulate([1, 2, 3, 4])))   # ví dụ accumulate: tổng dồn
print(before)

[1, 3, 6, 10]
[0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]


`itertools.accumulate(lst)` trả về các **tổng dồn**: `[a0, a0+a1, a0+a1+a2, ...]`. Thêm `0` ở đầu và bỏ phần tử cuối để `before[i]` là tổng **trước** tháng `i + 1`.

In [12]:
BEFORE = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]

def day_of_year_prefix(date: str) -> int:
    y, m, d = int(date[:4]), int(date[5:7]), int(date[8:])
    extra = 1 if m > 2 and is_leap(y) else 0
    return BEFORE[m - 1] + d + extra

for s in ["2019-01-09", "2019-02-10", "2004-03-01", "1900-03-01", "2004-02-29"]:
    print(s, "->", day_of_year_prefix(s))

2019-01-09 -> 9
2019-02-10 -> 41
2004-03-01 -> 61
1900-03-01 -> 60
2004-02-29 -> 60


Chú ý `m > 2`: ngày 29/2 năm nhuận (`m = 2`) **không** cộng thêm — chính ngày đó đã nằm trong `d = 29`.

## 8. Cách 4 — Dùng thư viện `datetime` (đối chiếu)

- `datetime.date(y, m, d)` tạo đối tượng ngày.
- `.timetuple().tm_yday` trả về đúng "ngày thứ mấy trong năm".
- Hoặc trừ hai ngày: `(date(y,m,d) - date(y,1,1)).days + 1`.

Thư viện đã cài sẵn quy tắc nhuận Gregorian, rất hợp để **kiểm tra** lời giải tự viết.

In [13]:
import datetime

def day_of_year_lib(date: str) -> int:
    y, m, d = map(int, date.split("-"))
    return datetime.date(y, m, d).timetuple().tm_yday

def day_of_year_sub(date: str) -> int:
    dt = datetime.date.fromisoformat(date)
    return (dt - datetime.date(dt.year, 1, 1)).days + 1

print(day_of_year_lib("2019-02-10"), day_of_year_sub("2019-02-10"))
print(datetime.date(2019, 2, 10) - datetime.date(2019, 1, 1))   # timedelta

41 41
40 days, 0:00:00


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def dayOfYear(self, date: str) -> int:
        y, m, d = int(date[:4]), int(date[5:7]), int(date[8:])
        days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
        if (y % 4 == 0 and y % 100 != 0) or y % 400 == 0:
            days[1] = 29
        return sum(days[:m - 1]) + d

sol = Solution()
print(sol.dayOfYear("2019-01-09"), sol.dayOfYear("2019-02-10"))

9 41


In [15]:
class SolutionPrefix:
    def dayOfYear(self, date: str) -> int:
        before = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
        y, m, d = map(int, date.split("-"))
        leap = (y % 4 == 0 and y % 100 != 0) or y % 400 == 0
        return before[m - 1] + d + (1 if leap and m > 2 else 0)

print(SolutionPrefix().dayOfYear("2004-03-01"), SolutionPrefix().dayOfYear("1900-03-01"))

61 60


## 10. Vì sao đúng?

Ngày `d` của tháng `m` đứng sau **toàn bộ** các ngày của tháng `1..m-1` và sau `d - 1` ngày đầu tháng `m`. Số ngày đứng trước nó trong năm là `sum(days[0 : m-1]) + (d - 1)`, nên thứ tự (đếm từ 1) là `sum(days[:m-1]) + d`.

Điểm duy nhất phụ thuộc năm là độ dài tháng 2, được xử lý bằng quy tắc nhuận Gregorian — áp dụng đúng cho cả 1900 (không nhuận) và 2000 (nhuận).

**Invariant trong trace:** sau khi cộng tháng `k`, `total` = số ngày từ 1/1 đến hết tháng `k`.

## 11. Độ phức tạp

Input có độ dài cố định 10 ký tự, có 12 tháng → mọi cách đều `O(1)`.
- Cộng các tháng trước: tối đa 11 phép cộng → `O(1)` thời gian, `O(1)` bộ nhớ (list 12 phần tử).
- Prefix sum: 1 phép tra bảng → `O(1)`.
- Đếm từng ngày: tối đa 366 bước → vẫn `O(1)` về mặt lý thuyết nhưng chậm hơn hằng số ~30 lần.

Nếu tổng quát hóa thành "số tháng = M", cách cộng là `O(M)`, prefix là `O(1)` mỗi truy vấn sau `O(M)` tiền xử lý.

## 12. Bộ test: mẫu đề, biên, bẫy năm nhuận, đối chiếu toàn miền

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS ({actual})")

sol = Solution()
check("đề 1", sol.dayOfYear("2019-01-09"), 9)
check("đề 2", sol.dayOfYear("2019-02-10"), 41)
check("ngày đầu năm", sol.dayOfYear("1900-01-01"), 1)
check("cuối năm thường", sol.dayOfYear("2019-12-31"), 365)
check("cuối năm nhuận", sol.dayOfYear("2016-12-31"), 366)
check("29/2 năm nhuận", sol.dayOfYear("2004-02-29"), 60)
check("1/3 năm nhuận", sol.dayOfYear("2004-03-01"), 61)
check("1900 không nhuận", sol.dayOfYear("1900-03-01"), 60)
check("2000 nhuận", sol.dayOfYear("2000-03-01"), 61)

đề 1                   PASS (9)
đề 2                   PASS (41)
ngày đầu năm           PASS (1)
cuối năm thường        PASS (365)
cuối năm nhuận         PASS (366)
29/2 năm nhuận         PASS (60)
1/3 năm nhuận          PASS (61)
1900 không nhuận       PASS (60)
2000 nhuận             PASS (61)


In [17]:
# Đối chiếu MỌI ngày từ 1900-01-01 tới 2019-12-31 với datetime
start = datetime.date(1900, 1, 1)
end = datetime.date(2019, 12, 31)
cur = start
count = 0
while cur <= end:
    s = cur.isoformat()
    expected = cur.timetuple().tm_yday
    assert Solution().dayOfYear(s) == expected
    assert SolutionPrefix().dayOfYear(s) == expected
    assert day_of_year(s) == expected
    cur += datetime.timedelta(days=1)
    count += 1
print("Đã kiểm tra", count, "ngày, tất cả khớp datetime. PASS")

Đã kiểm tra 43829 ngày, tất cả khớp datetime. PASS


In [18]:
# Bản đếm từng ngày chậm hơn, chỉ kiểm tra trên vài năm
for y in [1900, 1904, 2000, 2019]:
    cur = datetime.date(y, 1, 1)
    while cur.year == y:
        assert day_of_year_walk(cur.isoformat()) == cur.timetuple().tm_yday
        cur += datetime.timedelta(days=1)
print("day_of_year_walk khớp trên 4 năm mẫu. PASS")

day_of_year_walk khớp trên 4 năm mẫu. PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quy tắc nhuận thiếu điều kiện thế kỷ
```python
leap = y % 4 == 0          # 1900 bị coi là nhuận → "1900-03-01" ra 61 (SAI)
```

### Lỗi 2: Cộng cả tháng hiện tại
```python
sum(days[:m]) + d          # "2019-01-09" → 31 + 9 = 40 (SAI)
```

### Lỗi 3: Sửa list dùng chung (biến toàn cục)
```python
DAYS = [31, 28, ...]
if leap: DAYS[1] = 29      # các lần gọi sau với năm thường cũng bị 29!
```

### Lỗi 4: Cộng ngày nhuận khi `m <= 2` ở bản prefix
`"2004-02-10"` sẽ thành 42 thay vì 41.

In [19]:
def wrong_leap(date):
    y, m, d = map(int, date.split("-"))
    days = [31, 29 if y % 4 == 0 else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    return sum(days[:m - 1]) + d

def wrong_slice(date):
    y, m, d = map(int, date.split("-"))
    days = [31, 29 if is_leap(y) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    return sum(days[:m]) + d

SHARED = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
def wrong_shared(date):
    y, m, d = map(int, date.split("-"))
    if is_leap(y):
        SHARED[1] = 29          # sửa list dùng chung, không bao giờ trả lại 28
    return sum(SHARED[:m - 1]) + d

print("1900-03-01: đúng =", day_of_year("1900-03-01"), "| sai nhuận =", wrong_leap("1900-03-01"))
print("2019-01-09: đúng =", day_of_year("2019-01-09"), "| sai slice =", wrong_slice("2019-01-09"))
print("2019-03-01 trước:", wrong_shared("2019-03-01"))
wrong_shared("2004-03-01")
print("2019-03-01 sau khi gọi năm nhuận:", wrong_shared("2019-03-01"), "(đúng là 60)")

1900-03-01: đúng = 60 | sai nhuận = 61
2019-01-09: đúng = 9 | sai slice = 40
2019-03-01 trước: 60
2019-03-01 sau khi gọi năm nhuận: 61 (đúng là 60)


## 14. Mẹo nhận dạng pattern

- Bài **ngày tháng**: luôn hỏi ngay "năm nhuận xử lý thế nào?" và "đếm từ 0 hay từ 1?".
- "Vị trí của phần tử trong một chuỗi các khối có độ dài khác nhau" → **tổng các khối trước + vị trí trong khối**. Cùng ý: đổi (giờ, phút, giây) → tổng số giây.
- Nhiều truy vấn trên cùng bảng → **prefix sum** để trả lời `O(1)`.
- Dùng `datetime` để **kiểm thử**, còn khi phỏng vấn nên tự viết logic.

Bài liên quan: 1185 (Day of the Week), 1360 (Number of Days Between Two Dates).

## 15. Biến thể

### 15.1. Ngược lại: từ (năm, số thứ tự ngày) → `"YYYY-MM-DD"`
Trừ dần số ngày của từng tháng cho tới khi phần còn lại `<=` số ngày tháng hiện tại.

In [20]:
def date_from_day_of_year(y: int, k: int) -> str:
    days = [31, 29 if is_leap(y) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    m = 1
    while k > days[m - 1]:
        k -= days[m - 1]
        m += 1
    return f"{y:04d}-{m:02d}-{k:02d}"

print(date_from_day_of_year(2019, 41), date_from_day_of_year(2004, 60), date_from_day_of_year(2019, 365))
for s in ["2019-02-10", "2004-02-29", "1900-12-31"]:
    assert date_from_day_of_year(int(s[:4]), day_of_year(s)) == s
print("Đổi qua đổi lại khớp. PASS")

2019-02-10 2004-02-29 2019-12-31
Đổi qua đổi lại khớp. PASS


`f"{m:02d}"` định dạng số nguyên rộng 2 ký tự, thêm 0 ở đầu nếu thiếu: `2 → "02"`.

### 15.2. Số ngày từ một mốc cố định (ví dụ 1971-01-01)
Cộng số ngày của các năm trước + `day_of_year`. Đây là nền tảng cho bài 1185 và 1360.

In [21]:
def days_since_1971(date: str) -> int:
    y = int(date[:4])
    total = 0
    for year in range(1971, y):
        total += 366 if is_leap(year) else 365
    return total + day_of_year(date) - 1      # 1971-01-01 là ngày 0

print(days_since_1971("1971-01-01"), days_since_1971("1972-01-01"), days_since_1971("2019-08-31"))
assert days_since_1971("2019-08-31") == (datetime.date(2019, 8, 31) - datetime.date(1971, 1, 1)).days

0 365 17774


### 15.3. So sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Đếm từng ngày | mô phỏng lịch | `O(366)` | `O(1)` | dễ nghĩ, thừa việc |
| Cộng tháng trước | `sum(days[:m-1]) + d` | `O(12)` | `O(1)` | **lời giải chính** |
| Prefix sum | tra bảng `before` | `O(1)` | `O(1)` | nhanh nhất |
| `datetime` | thư viện | `O(1)` | `O(1)` | dùng để kiểm thử |

## 16. Bài tập vận dụng (TODO)

### Bài 1 — Tính tay
`"2000-12-31"` là ngày thứ mấy? `"2100-03-01"` (ngoài miền đề, nhưng hãy tính theo quy tắc)?

### Bài 2 — Viết `is_leap` của riêng bạn

In [22]:
def is_leap_exercise(y: int) -> bool:
    # TODO: 3 điều kiện %4, %100, %400
    pass

# assert [is_leap_exercise(y) for y in (1900, 2000, 2004, 2019)] == [False, True, True, False]

### Bài 3 — Hoàn thiện `day_of_year` bằng vòng lặp (không dùng `sum`)

In [23]:
def day_of_year_loop_exercise(date: str) -> int:
    # TODO 1: tách y, m, d
    # TODO 2: list số ngày, sửa tháng 2 nếu nhuận
    # TODO 3: for month in range(1, m): total += ...
    pass

# assert day_of_year_loop_exercise("2019-02-10") == 41
# assert day_of_year_loop_exercise("1900-03-01") == 60

### Bài 4 — Số ngày giữa hai ngày (LeetCode 1360)
Trả về giá trị tuyệt đối số ngày chênh lệch giữa hai ngày `"YYYY-MM-DD"` (năm 1971–2100).

In [24]:
def days_between_exercise(date1: str, date2: str) -> int:
    # TODO: quy cả hai về số ngày tính từ 1971-01-01, rồi lấy abs của hiệu
    pass

# assert days_between_exercise("2019-06-29", "2019-06-30") == 1
# assert days_between_exercise("2020-01-15", "2019-12-31") == 15

### Bài 5 — Ngày thứ k còn lại của năm
Còn bao nhiêu ngày nữa thì hết năm (không tính ngày hiện tại)?

In [25]:
def days_left_exercise(date: str) -> int:
    # TODO: (365 hoặc 366) - day_of_year(date)
    pass

# assert days_left_exercise("2019-12-31") == 0
# assert days_left_exercise("2016-01-01") == 365

### Bài 6 — Mini challenge: số thứ tự tuần
Tuần 1 gồm ngày 1..7 của năm, tuần 2 gồm 8..14, … Trả về số tuần của một ngày.

In [26]:
def week_number_exercise(date: str) -> int:
    # TODO: dùng day_of_year và phép chia nguyên
    pass

# assert week_number_exercise("2019-01-07") == 1
# assert week_number_exercise("2019-01-08") == 2
# assert week_number_exercise("2019-12-31") == 53

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | 2000 có nhuận không? | 2100 chia hết 100 nhưng không chia hết 400 | 366 và 31+28+1 |
| 2 | ưu tiên điều kiện 400 | `or` / `and` | `(y%4==0 and y%100!=0) or y%400==0` |
| 3 | cộng dồn | `total = 0` | `total += days[month - 1]` rồi `+ d` |
| 4 | về một mốc chung | hàm `days_since_1971` | `abs(a - b)` |
| 5 | độ dài năm | `366 if is_leap(y) else 365` | trừ `day_of_year` |
| 6 | 7 ngày một tuần | đổi về đếm từ 0 | `(day_of_year - 1) // 7 + 1` |

## 18. Đáp án tham khảo

**Bài 1:** `"2000-12-31"` → **366** (2000 nhuận vì chia hết cho 400). `"2100-03-01"` → `31 + 28 + 1 = ` **60** (2100 không nhuận).

In [27]:
def is_leap_answer(y: int) -> bool:
    return (y % 4 == 0 and y % 100 != 0) or y % 400 == 0

def day_of_year_loop_answer(date: str) -> int:
    y, m, d = map(int, date.split("-"))
    days = [31, 29 if is_leap_answer(y) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    total = 0
    for month in range(1, m):
        total += days[month - 1]
    return total + d

def days_between_answer(date1: str, date2: str) -> int:
    return abs(days_since_1971(date1) - days_since_1971(date2))

def days_left_answer(date: str) -> int:
    y = int(date[:4])
    return (366 if is_leap_answer(y) else 365) - day_of_year(date)

def week_number_answer(date: str) -> int:
    return (day_of_year(date) - 1) // 7 + 1

assert day_of_year("2000-12-31") == 366
assert day_of_year("2100-03-01") == 60
assert [is_leap_answer(y) for y in (1900, 2000, 2004, 2019)] == [False, True, True, False]
assert day_of_year_loop_answer("2019-02-10") == 41
assert day_of_year_loop_answer("1900-03-01") == 60
assert days_between_answer("2019-06-29", "2019-06-30") == 1
assert days_between_answer("2020-01-15", "2019-12-31") == 15
assert days_left_answer("2019-12-31") == 0
assert days_left_answer("2016-01-01") == 365
assert week_number_answer("2019-01-07") == 1
assert week_number_answer("2019-01-08") == 2
assert week_number_answer("2019-12-31") == 53
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def dayOfYear(self, date: str) -> int:
        y, m, d = int(date[:4]), int(date[5:7]), int(date[8:])
        days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
        if (y % 4 == 0 and y % 100 != 0) or y % 400 == 0:
            days[1] = 29
        return sum(days[:m - 1]) + d
```

**Checklist:**
- [ ] Quy tắc nhuận đủ **3 điều kiện** (1900 không nhuận, 2000 nhuận)?
- [ ] Cộng các tháng **trước** (`days[:m - 1]`), không cộng tháng hiện tại?
- [ ] Không sửa list dùng chung giữa các lần gọi?
- [ ] Test `"…-01-01"` → 1, `"…-12-31"` → 365/366, 29/2 và 1/3 năm nhuận?